In [19]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


# =====================
# CSV読み込み
# =====================

df = pd.read_csv(
    "../one_sentence_stylometry_features.csv"
)

df = df.fillna(0)


# =====================
# 文書名取得
# =====================

# Human_text_01.txt_s1
# ↓
# Human_text_01.txt

df["doc_id"] = (
    df["filename"]
    .str.replace(
        r"_s\d+$",
        "",
        regex=True
    )
)


# 文番号
# 各文書ごとに s1,s2,s3...
df["sent_order"] = (
    df.groupby("doc_id")
      .cumcount()
    +1
)


print(df[["filename","doc_id","sent_order"]].head())


# =====================
# 特徴量
# =====================

features = [
    c for c in df.columns
    if c not in [
        "filename",
        "doc_id",
        "sent_order",
        "label"
    ]
]


# =====================
# 学習文書
# =====================

# =====================
# 学習文書（★手動で並べずに自動生成します）
# =====================
# 1から13まで（rangeの終わりは指定＋1にするので14）を、2桁の0埋め文字列にする
base_numbers = [f"{i:02d}" for i in range(1, 21)]

# 1文ずつ増やして追加していく対象の文書
increment_number = "21"

# =====================
# 評価文書
# =====================

all_train_nums = base_numbers + [increment_number]
test_df = df[~df["doc_id"].str.contains("|".join(all_train_nums))]

# =====================
# 実験準備：ベースデータの抽出（各ペアごとに最小文数に揃える！）
# =====================
base_frames = []
for num in base_numbers:
    human_doc = f"Human_text_{num}.txt"
    ai_doc = f"ChatGPT_text_{num}.txt"
    
    # このペア（例: 02のHumanとAI）の間での最小文数を計算
    pair_max = df[df["doc_id"].isin([human_doc, ai_doc])].groupby("doc_id")["sent_order"].max().min()
    
    # 揃えた文数分だけデータを抽出
    pair_df = df[df["doc_id"].isin([human_doc, ai_doc]) & (df["sent_order"] <= pair_max)]
    base_frames.append(pair_df)

# 各ペアがそれぞれ綺麗に揃った状態のベースデータ
# 【★ここを修正】base_frames が空でなければ結合し、空なら空の DataFrame を作る
if base_frames:
    base_df = pd.concat(base_frames, ignore_index=True)
else:
    base_df = pd.DataFrame() # 空のデータフレーム

# =====================
# s1,s1+s2...実験
# =====================
results = []

inc_docs = [f"Human_text_{increment_number}.txt", f"ChatGPT_text_{increment_number}.txt"]
# 追加する文書（04）の共通の最大文数を取得
max_sent = df[df["doc_id"].isin(inc_docs)].groupby("doc_id")["sent_order"].max().min()

for n in range(1, max_sent + 1):
    print("\n================")
    print(f"ベース(01〜{base_numbers[-1] if base_numbers else 'なし'}) + {increment_number} の s0〜s{n - 1} 使用")
    print("================")
    
    # 追加文書（今回なら01）は「n文目まで」取得
    inc_df = df[(df["doc_id"].isin(inc_docs)) & (df["sent_order"] <= n)]
    
    # 【★ここを修正】ベース（base_df）が空の場合は、追加文書（inc_df）だけで学習データを作る
    if len(base_df) > 0:
        train_df = pd.concat([base_df, inc_df], ignore_index=True)
    else:
        train_df = inc_df # 01のみの場合はこれだけを使う
    
    # 件数確認
    print(train_df["label"].value_counts())
    
    X_train = train_df[features]
    y_train = train_df["label"]
    
    X_test = test_df[features]
    y_test = test_df["label"]
    
    # 標準化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Logistic Regression
    model = LogisticRegression(max_iter=5000, class_weight="balanced")
    model.fit(X_train, y_train)
    
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    
    print("Accuracy:", acc)
    print(classification_report(y_test, pred, zero_division=0))
    
    results.append({"used_sentence": n, "accuracy": acc})
    
# # =====================
# # 特徴量の寄与度（係数）の取得と履歴保存
# # =====================
# coefficients = model.coef_[0]

# df_importance = pd.DataFrame({
#     "Feature": features,
#     "Coefficient": coefficients
# })
# df_importance["Abs_Coefficient"] = df_importance["Coefficient"].abs()
# df_importance = df_importance.sort_values(by="Abs_Coefficient", ascending=False).reset_index(drop=True)

# # 【追加】どの実験条件のデータかを識別する列を追加
# df_importance["base_docs"] = ", ".join(base_numbers) if base_numbers else "None"
# df_importance["increment_doc"] = increment_number

# # 列の並びを整える
# cols = ["base_docs", "increment_doc", "Feature", "Coefficient", "Abs_Coefficient"]
# df_importance = df_importance[cols]

# imp_csv_filename = "feature_importance_result.csv"

# try:
#     # 既存のCSVを読み込む
#     df_imp_existing = pd.read_csv(imp_csv_filename)
    
#     # 同じ実験条件がすでにあれば古いデータを削除（上書き）
#     cond = (df_imp_existing["base_docs"] == (", ".join(base_numbers) if base_numbers else "None")) & \
#            (df_imp_existing["increment_doc"] == increment_number)
#     df_imp_existing = df_imp_existing[~cond]
    
#     # 結合
#     df_imp_combined = pd.concat([df_imp_existing, df_importance], ignore_index=True)
#     print(f"\n既存の {imp_csv_filename} に特徴量の寄与度を上書き・追記しました。")

# except (FileNotFoundError, KeyError):
#     # ファイルがない、または古い形式の場合は新規作成
#     df_imp_combined = df_importance
#     print(f"\n新しく {imp_csv_filename} を作成しました。")

# # 保存
# df_imp_combined.to_csv(imp_csv_filename, index=False)
# # =====================
# # 実験結果のデータフレーム化と保存（追記・上書き対応）
# # =====================
# # 結果をDataFrameに変換し、実験条件のカラムを追加
# df_new_results = pd.DataFrame(results)
# df_new_results["base_docs"] = ", ".join(base_numbers)       # 例: "01, 02, 03"
# df_new_results["increment_doc"] = increment_number         # 例: "06"

# # カラムの並び順を綺麗に整える
# cols = ["base_docs", "increment_doc", "used_sentence", "accuracy"]
# df_new_results = df_new_results[cols]

# csv_filename = "s1_increment_result.csv"

# try:
#     # 既存のCSVファイルがあるか確認して読み込む
#     df_existing = pd.read_csv(csv_filename)
    
#     # 全く同じ実験条件（同じbase_docsとincrement_doc）がすでにあれば、古いデータを削除（上書き）
#     condition = (df_existing["base_docs"] == ", ".join(base_numbers)) & (df_existing["increment_doc"] == increment_number)
#     df_existing = df_existing[~condition]
    
#     # 既存のデータと今回の新しいデータを結合
#     df_combined = pd.concat([df_existing, df_new_results], ignore_index=True)
#     print(f"\n既存の {csv_filename} に今回の実験結果を上書き・追記しました。")

# except FileNotFoundError:
#     # 初めてCSVを作る場合は、今回の結果をそのまま使う
#     df_combined = df_new_results
#     print(f"\n新しく {csv_filename} を作成して保存しました。")

# # CSVファイルへ保存
# df_combined.to_csv(csv_filename, index=False)

               filename             doc_id  sent_order
0  Human_text_01.txt_s0  Human_text_01.txt           1
1  Human_text_01.txt_s1  Human_text_01.txt           2
2  Human_text_01.txt_s2  Human_text_01.txt           3
3  Human_text_01.txt_s3  Human_text_01.txt           4
4  Human_text_01.txt_s4  Human_text_01.txt           5

ベース(01〜20) + 21 の s0〜s0 使用
label
Human    232
AI       232
Name: count, dtype: int64
Accuracy: 0.7107843137254902
              precision    recall  f1-score   support

          AI       0.74      0.72      0.73       109
       Human       0.68      0.71      0.69        95

    accuracy                           0.71       204
   macro avg       0.71      0.71      0.71       204
weighted avg       0.71      0.71      0.71       204


ベース(01〜20) + 21 の s0〜s1 使用
label
Human    233
AI       233
Name: count, dtype: int64
Accuracy: 0.7107843137254902
              precision    recall  f1-score   support

          AI       0.74      0.72      0.73       109
    

In [65]:
import os
if os.path.exists("random_LR_Cont_all_increment_results_summary.csv"):
    os.remove("random_LR_Cont_all_increment_results_summary.csv")
    print("CSVファイルを強制削除しました。")
else:
    print("ファイルが見つかりません。")

CSVファイルを強制削除しました。


In [53]:
import os
if os.path.exists("feature_importance_result.csv"):
    os.remove("feature_importance_result.csv")
    print("CSVファイルを強制削除しました。")
else:
    print("ファイルが見つかりません。")

CSVファイルを強制削除しました。


In [84]:
import os
import pandas as pd
import numpy as np
import random
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

LIMIT_MAX_SENTENCE = True  # Trueで最大10文に制限、Falseで制限なし
# =====================
# 1. CSV読み込み
# =====================
df = pd.read_csv("../one_sentence_stylometry_features.csv")
df = df.fillna(0)

# =====================
# 2. 文書名・文番号の整理
# =====================
# 例: Human_text_01.txt_s1 -> Human_text_01.txt
df["doc_id"] = df["filename"].str.replace(r"_s\d+$", "", regex=True)

# 各文書ごとに s1, s2, s3... と番号を振る
df["sent_order"] = df.groupby("doc_id").cumcount() + 1

print("--- データフレームの先頭確認 ---")
print(df[["filename", "doc_id", "sent_order"]].head())


# ==========================================
# 3. 特徴量リストの定義（スイッチング用）
# ==========================================
function_features = [
    "機能語率",
    "ひらがな率",
    "句点頻度",
    "読点頻度",
    "助詞割合",
    "助動詞割合",
    "連体詞割合",
    "接続詞割合",
    "感動詞割合",


    
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か", "助詞_が", "助詞_で", "助詞_や", "助詞_を",
    "助詞_の", "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から", "助詞_たり", "助詞_など", "助詞_ながら",
    "助詞_だけ", "助詞_ば", "助詞_へ", "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら", "助詞_より",
    "助詞_こそ", "助詞_つつ", "助詞_くらい", "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率",

    "動詞割合",
    "名詞割合",
    "形容詞割合", 
    "副詞割合", 
    "形状詞割合",
    "漢字率",
    "カタカナ率",
    "英数字率"
]

# ------------------------------------------
# ★ 実験したい特徴量セットをここで選定
# ------------------------------------------
feature_mode = "Function"  # "All" or "Function" or "Content"

if feature_mode == "All":
    base_features = function_features + content_features
elif feature_mode == "Function":
    base_features = function_features
elif feature_mode == "Content":
    base_features = content_features

print(f"\n【実験設定】選択された特徴量モード: {feature_mode} (ベース特徴量数: {len(base_features)})")

import random

# ==========================================
# 4. 5. 6. 【修正版】全自動実験ループ（30文書すべて学習＋新規テストデータ版）
# ==========================================

# 実験結果をすべて格納するリスト
all_experiments_results = []

# --- 1. 新規テストデータの読み込みと前処理 ---
# ★新規テストデータのCSVファイル名を指定してください
test_csv_path = "test_one_sentence_stylometry_features.csv" 

if os.path.exists(test_csv_path):
    raw_test_df = pd.read_csv(test_csv_path).fillna(0)
    raw_test_df["doc_id"] = raw_test_df["filename"].str.replace(r"_s\d+$", "", regex=True)
    raw_test_df["sent_order"] = raw_test_df.groupby("doc_id").cumcount() + 1
    test_df = raw_test_df.copy()
    print("[確認] 新規テストデータの読み込みに成功しました。")
else:
    raise FileNotFoundError(f"新規テストデータが見つかりません: {test_csv_path}")


# --- 2. 既存の30文書をすべて学習（インクリメント）に回す設定 ---
random.seed(41) 
shuffled_numbers = random.sample(list(range(1, 31)), 30)  # 1〜30番をすべてシャッフル

print(f"【学習・追加に使う全30文書（順序ランダム）】: {[f'{i:02d}' for i in shuffled_numbers]}")


# --- 【外側ループ】シャッフルされた30個のリストを順に処理 ---
for idx in range(30):
    
    # 今回インクリメント（追加）対象となる数値
    inc_raw_num = shuffled_numbers[idx]
    increment_number = f"{inc_raw_num:02d}"
    
    # それまでに処理した数値をすべてベースにする
    base_numbers = [f"{shuffled_numbers[i]:02d}" for i in range(idx)]
    
    base_display = ", ".join(base_numbers) if base_numbers else "なし"
    print(f"\n========================================================")
    print(f"【自動実験】ベース: [{base_display}]  ===> インクリメント: {increment_number}")
    print(f"========================================================")
    
    # 各ペアごとに最小文数に揃えてベースデータを抽出
    base_frames = []
    for num in base_numbers:
        human_doc = f"Human_text_{num}.txt"
        ai_doc = f"ChatGPT_text_{num}.txt"
        pair_max = df[df["doc_id"].isin([human_doc, ai_doc])].groupby("doc_id")["sent_order"].max().min()
        pair_df = df[df["doc_id"].isin([human_doc, ai_doc]) & (df["sent_order"] <= pair_max)]
        base_frames.append(pair_df)
        
    if base_frames:
        base_df = pd.concat(base_frames, ignore_index=True)
    else:
        base_df = pd.DataFrame()
        
    inc_docs = [f"Human_text_{increment_number}.txt", f"ChatGPT_text_{increment_number}.txt"]
    max_sent = df[df["doc_id"].isin(inc_docs)].groupby("doc_id")["sent_order"].max().min()
    
    # --------------------------------------------------
    # ★追加：LIMIT_MAX_SENTENCE が True なら max_sent を最大10に制限する
    # --------------------------------------------------
    if LIMIT_MAX_SENTENCE:
        max_sent = min(max_sent, 10)
    # --------------------------------------------------
    
    # --- 【内側ループ】s1 〜 sn まで文数を増やしていくループ ---
    for n in range(1, max_sent + 1):
        inc_df = df[(df["doc_id"].isin(inc_docs)) & (df["sent_order"] <= n)]
        if len(base_df) > 0:
            current_raw_df = pd.concat([base_df, inc_df], ignore_index=True)
        else:
            current_raw_df = inc_df
            
        # 訓練データの横持ち変換
        X_list, y_list = [], []
        for doc_id, group in current_raw_df.groupby("doc_id"):
            label = group["label"].iloc[0]
            doc_features = []
            for i in range(1, n + 1):
                sent_row = group[group["sent_order"] == i]
                sent_vals = sent_row[base_features].values[0] if not sent_row.empty else np.zeros(len(base_features))
                doc_features.extend(sent_vals)
            X_list.append(doc_features)
            y_list.append(label)
            
        X_train = np.array(X_list)
        y_train = np.array(y_list)
        
        for col in base_features:
            if col not in test_df.columns:
                test_df[col] = 0.0
                
        # 新規テストデータの横持ち変換
        X_test_list, y_test_list = [], []
        for doc_id, group in test_df.groupby("doc_id"):
            if group["sent_order"].max() < n:
                continue
            label = group["label"].iloc[0]
            doc_features = []
            for i in range(1, n + 1):
                sent_row = group[group["sent_order"] == i]
                doc_features.extend(sent_row[base_features].values[0])
            X_test_list.append(doc_features)
            y_test_list.append(label)
            
        X_test = np.array(X_test_list)
        y_test = np.array(y_test_list)
        
        # データが空の場合はスキップ
        if len(X_test) == 0:
            continue
            
        # 標準化 & 学習
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, pred)
        
        all_experiments_results.append({
            "base_docs": ", ".join(base_numbers) if base_numbers else "None",
            "increment_doc": increment_number,
            "used_sentences": f"s1-s{n}",
            "accuracy": acc
        })

# ==========================================
# 7. 全結果をCSVに一括保存
# ==========================================
df_results = pd.DataFrame(all_experiments_results)
df_results.to_csv(f"random_LR_{feature_mode}_all_increment_results_summary.csv", index=False)
print(f"\n[完了] 30文書すべてを学習に使用した実験が終了し、'random_LR_{feature_mode}_all_increment_results_summary.csv' に保存されました！")

--- データフレームの先頭確認 ---
               filename             doc_id  sent_order
0  Human_text_01.txt_s0  Human_text_01.txt           1
1  Human_text_01.txt_s1  Human_text_01.txt           2
2  Human_text_01.txt_s2  Human_text_01.txt           3
3  Human_text_01.txt_s3  Human_text_01.txt           4
4  Human_text_01.txt_s4  Human_text_01.txt           5

【実験設定】選択された特徴量モード: Function (ベース特徴量数: 39)
[確認] 新規テストデータの読み込みに成功しました。
【学習・追加に使う全30文書（順序ランダム）】: ['13', '11', '08', '06', '30', '19', '23', '10', '18', '09', '26', '25', '01', '28', '22', '17', '03', '14', '12', '27', '20', '05', '16', '15', '21', '07', '04', '02', '29', '24']

【自動実験】ベース: [なし]  ===> インクリメント: 13

【自動実験】ベース: [13]  ===> インクリメント: 11

【自動実験】ベース: [13, 11]  ===> インクリメント: 08

【自動実験】ベース: [13, 11, 08]  ===> インクリメント: 06

【自動実験】ベース: [13, 11, 08, 06]  ===> インクリメント: 30

【自動実験】ベース: [13, 11, 08, 06, 30]  ===> インクリメント: 19

【自動実験】ベース: [13, 11, 08, 06, 30, 19]  ===> インクリメント: 23

【自動実験】ベース: [13, 11, 08, 06, 30, 19, 23]  ===> インクリメント: 10

【自動実験】ベー

In [90]:
import os
import pandas as pd
import numpy as np
import random
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# ★ハッシュタグで最大文数の制限をオンオフ
# ==========================================
LIMIT_MAX_SENTENCE = False  # Trueで最大10文に制限、Falseで制限なし
# ==========================================

# =====================
# 1. 訓練データの読み込み
# =====================
df = pd.read_csv("../one_sentence_stylometry_features.csv")
df = df.fillna(0)

# 訓練データ用の doc_id 作成
df["doc_id"] = df["filename"].str.replace(r"_s\d+$", "", regex=True)

# 各文書ごとに s1, s2, s3... と番号を振る
df["sent_order"] = df.groupby("doc_id").cumcount() + 1

# ==========================================
# 2. テストデータの読み込み（1行1文書のまとまったファイルを使用）
# ==========================================
test_csv_path = "stylometry_features_test.csv"

if os.path.exists(test_csv_path):
    # テストデータはそのまま読み込む
    test_df = pd.read_csv(test_csv_path).fillna(0)
    test_df["doc_id"] = test_df["filename"].str.replace(r"_s\d+$", "", regex=True)
    print(f"[確認] テストデータ（{test_csv_path}）の読み込みに成功しました。行数: {len(test_df)}")
else:
    raise FileNotFoundError(f"指定されたテスト用ファイルが見つかりません: {test_csv_path}")

# ==========================================
# 3. 特徴量リストの定義
# ==========================================
function_features = [
    "機能語率", "ひらがな率", "句点頻度", "読点頻度", "助詞割合", "助動詞割合", "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か", "助詞_が", "助詞_で", "助詞_や", "助詞_を", "助詞_の",
    "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から", "助詞_たり", "助詞_など", "助詞_ながら", "助詞_だけ", "助詞_ば",
    "助詞_へ", "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら", "助詞_より", "助詞_こそ", "助詞_つつ", "助詞_くらい",
    "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合", "形容詞割合", "副詞割合", "形状詞割合", "漢字率", "カタカナ率", "英数字率"
]

# 実験したい特徴量セットを選定
feature_mode = "Content"  # "All" or "Function" or "Content"

if feature_mode == "All":
    base_features = function_features + content_features
elif feature_mode == "Function":
    base_features = function_features
elif feature_mode == "Content":
    base_features = content_features

print(f"\n【実験設定】選択された特徴量モード: {feature_mode} (ベース特徴量数: {len(base_features)})")

# テストデータ側に存在しないカラムを 0.0 で作成してエラーを防止
for col in base_features:
    if col not in test_df.columns:
        test_df[col] = 0.0

# ==========================================
# 4. 30文書のシャッフル設定（インクリメント学習）
# ==========================================
random.seed(41)
shuffled_numbers = random.sample(list(range(1, 31)), 30)

all_experiments_results = []

# --- 【外側ループ】30文書を1つずつ学習データに追加していく ---
for idx in range(30):
    inc_raw_num = shuffled_numbers[idx]
    increment_number = f"{inc_raw_num:02d}"
    
    base_numbers = [f"{shuffled_numbers[i]:02d}" for i in range(idx)]
    base_display = ", ".join(base_numbers) if base_numbers else "なし"
    print(f"\n========================================================")
    print(f"【自動実験】ベース: [{base_display}]  ===> インクリメント: {increment_number}")
    print(f"========================================================")
    
    # ベースに使う文書リスト
    target_train_nums = base_numbers + [increment_number]
    
    # 今回追加されるインクリメント文書ペアの最小文数を計算
    inc_docs = [f"Human_text_{increment_number}.txt", f"ChatGPT_text_{increment_number}.txt"]
    max_sent = df[df["doc_id"].isin(inc_docs)].groupby("doc_id")["sent_order"].max().min()
    
    # LIMIT_MAX_SENTENCE が True なら max_sent を最大10に制限する
    if LIMIT_MAX_SENTENCE:
        max_sent = min(max_sent, 10)
        
    # --- 【内側ループ】s1 〜 sn まで訓練データの文数を増やしていくループ ---
    for n in range(1, max_sent + 1):
        print(f"  --> 現在の対象範囲: s1 - s{n}")
        
        train_frames = []
        for num in target_train_nums:
            h_doc = f"Human_text_{num}.txt"
            c_doc = f"ChatGPT_text_{num}.txt"
            
            # 各ペアごとに、1文目からn文目までを抽出
            pair_df = df[df["doc_id"].isin([h_doc, c_doc]) & (df["sent_order"] <= n)]
            train_frames.append(pair_df)
            
        current_train_df = pd.concat(train_frames, ignore_index=True)
        
        # ★重要: 抽出した「s1〜sn」の文データから、文書（doc_id）ごとに平均値を算出
        train_grouped = current_train_df.groupby("doc_id").agg({
            "label": "first",
            **{col: "mean" for col in base_features}
        }).reset_index()
        
        # 特徴量とラベルの切り出し
        X_train = train_grouped[base_features].values
        y_train = train_grouped["label"].values
        
        # テストデータ（1行1文書）をそのまま使用
        X_test = test_df[base_features].values
        y_test = test_df["label"].values
        
        # 標準化 & 学習
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
        model.fit(X_train_scaled, y_train)
        
        pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, pred)
        
        # 結果の保存
        all_experiments_results.append({
            "base_docs": ", ".join(base_numbers) if base_numbers else "None",
            "increment_doc": increment_number,
            "used_sentences": f"s1-s{n}",
            "accuracy": acc
        })

# ==========================================
# 5. 結果をCSVに一括保存
# ==========================================
df_results = pd.DataFrame(all_experiments_results)
df_results.to_csv(f"test-doc_random_LR_{feature_mode}_all_increment_results_summary.csv", index=False)
print(f"\n[完了] 実験が終了し、'random_LR_{feature_mode}_all_increment_results_summary.csv' に保存されました。")

[確認] テストデータ（stylometry_features_test.csv）の読み込みに成功しました。行数: 20

【実験設定】選択された特徴量モード: Content (ベース特徴量数: 9)

【自動実験】ベース: [なし]  ===> インクリメント: 13
  --> 現在の対象範囲: s1 - s1
  --> 現在の対象範囲: s1 - s2
  --> 現在の対象範囲: s1 - s3
  --> 現在の対象範囲: s1 - s4
  --> 現在の対象範囲: s1 - s5
  --> 現在の対象範囲: s1 - s6
  --> 現在の対象範囲: s1 - s7
  --> 現在の対象範囲: s1 - s8
  --> 現在の対象範囲: s1 - s9
  --> 現在の対象範囲: s1 - s10

【自動実験】ベース: [13]  ===> インクリメント: 11
  --> 現在の対象範囲: s1 - s1
  --> 現在の対象範囲: s1 - s2
  --> 現在の対象範囲: s1 - s3
  --> 現在の対象範囲: s1 - s4
  --> 現在の対象範囲: s1 - s5
  --> 現在の対象範囲: s1 - s6
  --> 現在の対象範囲: s1 - s7
  --> 現在の対象範囲: s1 - s8
  --> 現在の対象範囲: s1 - s9
  --> 現在の対象範囲: s1 - s10
  --> 現在の対象範囲: s1 - s11
  --> 現在の対象範囲: s1 - s12

【自動実験】ベース: [13, 11]  ===> インクリメント: 08
  --> 現在の対象範囲: s1 - s1
  --> 現在の対象範囲: s1 - s2
  --> 現在の対象範囲: s1 - s3
  --> 現在の対象範囲: s1 - s4
  --> 現在の対象範囲: s1 - s5
  --> 現在の対象範囲: s1 - s6
  --> 現在の対象範囲: s1 - s7
  --> 現在の対象範囲: s1 - s8
  --> 現在の対象範囲: s1 - s9
  --> 現在の対象範囲: s1 - s10
  --> 現在の対象範囲: s1 - s11
  --> 現在の対象範囲: s1 - s12
